<a href="https://colab.research.google.com/github/PranavMurari-git/nanogpt_from_scratch/blob/main/nanogpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import requests

# 1. Download the Tiny Shakespeare dataset
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(requests.get(url).text)

# 2. Read the text file
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Dataset size: {len(text)} characters")

# 3. Build the Vocabulary (every unique character in the text)
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Our Vocabulary: {''.join(chars)}")
print(f"Vocabulary Size: {vocab_size} unique characters")

# 4. Create the Tokenizer mapping (String-to-Integer and Integer-to-String)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Lambda functions to encode and decode text
encode = lambda s: [stoi[c] for c in s] # string to list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # list of ints to string

# --- Let's test it out ---
test_phrase = "O Romeo, Romeo!"
encoded_phrase = encode(test_phrase)

print(f"\nOriginal: '{test_phrase}'")
print(f"Encoded:  {encoded_phrase}")
print(f"Decoded:  '{decode(encoded_phrase)}'")

Dataset size: 1115394 characters
Our Vocabulary: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary Size: 65 unique characters

Original: 'O Romeo, Romeo!'
Encoded:  [27, 1, 30, 53, 51, 43, 53, 6, 1, 30, 53, 51, 43, 53, 2]
Decoded:  'O Romeo, Romeo!'


In [17]:
import torch

# 1. Convert our entire dataset into a PyTorch Tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data Tensor Shape: {data.shape}")
print(f"First 50 characters encoded: \n{data[:50]}\n")

# 2. Split the data (90% for training, 10% for validation)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# 3. Set our hyperparameters for batching
block_size = 8  # Maximum context length for predictions (X)
batch_size = 4  # How many independent sequences to process in parallel

# 4. The Batching Function
def get_batch(split):
    # Choose whether to pull from train or validation data
    data_source = train_data if split == 'train' else val_data

    # Generate random starting points (indices) for our batch
    ix = torch.randint(len(data_source) - block_size, (batch_size,))

    # Stack the context chunks (X) and the target characters (Y)
    x = torch.stack([data_source[i : i + block_size] for i in ix])
    y = torch.stack([data_source[i + 1 : i + block_size + 1] for i in ix])

    # Move the tensors to the GPU if available
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    x, y = x.to(device), y.to(device)

    return x, y

# --- Let's test the batcher ---
xb, yb = get_batch('train')
print(f"Inputs (X) shape: {xb.shape}")
print(f"Targets (Y) shape: {yb.shape}\n")

print("Let's look at the very first batch row:")
print(f"Context (X): {xb[0].tolist()} -> '{decode(xb[0].tolist())}'")
print(f"Target  (Y): {yb[0].tolist()} -> '{decode(yb[0].tolist())}'")

Data Tensor Shape: torch.Size([1115394])
First 50 characters encoded: 
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])

Inputs (X) shape: torch.Size([4, 8])
Targets (Y) shape: torch.Size([4, 8])

Let's look at the very first batch row:
Context (X): [56, 53, 61, 52, 6, 0, 20, 47] -> 'rown,
Hi'
Target  (Y): [53, 61, 52, 6, 0, 20, 47, 57] -> 'own,
His'


In [18]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Create a lookup table of size (65, 65).
        # Every token ID looks up a row of 65 numbers representing the "score" for the next token.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx is our (Batch, Time) tensor of input contexts (xb)
        # Logits are the raw predictions the model makes. Shape: (Batch, Time, Channels/Vocab)
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            # PyTorch's cross_entropy requires a specific shape, so we reshape our tensors
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            # Calculate how wrong the model is
            loss = F.cross_entropy(logits, targets)

        return logits, loss

# 1. Initialize the model
m = BigramLanguageModel(vocab_size)

# 2. Move the model to the GPU (crucial for speed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
m = m.to(device)

# 3. Pass our previous batch (xb, yb) through the model
logits, loss = m(xb, yb)

print(f"Prediction (Logits) shape: {logits.shape}")
print(f"Initial Loss: {loss.item():.4f}")

Prediction (Logits) shape: torch.Size([32, 65])
Initial Loss: 4.7726


In [19]:
# 1. Create a PyTorch Optimizer
# lr is the learning rate (how big of a step we take). 1e-3 is a good starting point.
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

# 2. The Training Loop
batch_size = 32 # We can process more data at once now

print("Starting training...")
for steps in range(10000): # Train for 10,000 steps

    # Grab a new batch of data
    xb, yb = get_batch('train')

    # Forward pass: Evaluate the model and calculate loss
    logits, loss = m(xb, yb)

    # Backward pass: Update the weights
    optimizer.zero_grad(set_to_none=True) # 1. Clear out old gradients from the last step
    loss.backward()                       # 2. Calculate the new gradients
    optimizer.step()                      # 3. Nudge the weights in the embedding table

    # Print our progress every 1000 steps
    if steps % 1000 == 0:
        print(f"Step {steps:4d} | Loss: {loss.item():.4f}")

print(f"Final Loss: {loss.item():.4f}")

Starting training...
Step    0 | Loss: 4.6076
Step 1000 | Loss: 3.6220
Step 2000 | Loss: 3.1012
Step 3000 | Loss: 2.8067
Step 4000 | Loss: 2.5969
Step 5000 | Loss: 2.5597
Step 6000 | Loss: 2.5768
Step 7000 | Loss: 2.5424
Step 8000 | Loss: 2.5535
Step 9000 | Loss: 2.3722
Final Loss: 2.5111


In [20]:
# Create a starting token: A single zero (which happens to be the newline character)
# Shape is (1, 1) -> 1 batch, 1 time step
idx = torch.zeros((1, 1), dtype=torch.long, device=device)

# Generate 300 characters
generated_tokens = []
for _ in range(300):
    # Get the predictions for the current context
    logits, _ = m(idx)

    # Focus only on the very last time step
    logits = logits[:, -1, :] # becomes (Batch, Channels)

    # Convert the raw logits into probabilities using softmax
    probs = F.softmax(logits, dim=-1)

    # Sample from the probability distribution to get the next character
    idx_next = torch.multinomial(probs, num_samples=1)

    # Append it to our list for later decoding
    generated_tokens.append(idx_next.item())

    # Update the context by appending the new character
    idx = torch.cat((idx, idx_next), dim=1)

# Decode the generated integer list back into a string
print(decode(generated_tokens))


PENathit icche ied,

wfenk, outhe.
y INGrthe mo's,
Towig hedaverd m'd ad wendsthee os hor SI t tor'd, may ang l! seend andeen? t assthy
Prckeslon ntouie ramas,
Kwio o m atilfu ay ve as kil y, are ist waveistiltr meaf t
GHMau, tyfioman;
Cackisthendoshad.
dard oor doroury thor f s ther greavendigrrme


In [21]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# We will define some hyper-parameters for our architecture
n_embd = 32      # The size of our embedding vectors
head_size = 16   # The size of the Query, Key, and Value vectors
block_size = 8   # The maximum context length (from our data batching step)

class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        # Linear layers to project our embeddings into Q, K, and V
        # bias=False because it's standard practice in attention to just multiply the weights
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # 'tril' is our lower triangular matrix for the causal mask.
        # We use register_buffer because it is a constant matrix, not a learned weight.
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape # Batch, Time (context length), Channels (embedding dim)

        # 1. Emit Queries and Keys
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # 2. Compute Attention Scores (Affinities)
        # q @ k^T mathematically checks how well every query matches every key.
        # We scale by C**-0.5 (the square root of the dimension) to keep the variance in check.
        wei = q @ k.transpose(-2, -1) * (C ** -0.5) # (B, T, T)

        # 3. Apply the Causal Mask (The "No Cheating" rule)
        # We replace all 0s in our lower triangular matrix with negative infinity.
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # 4. Softmax to get probabilities (Negative infinities become exactly 0.0)
        wei = F.softmax(wei, dim=-1) # (B, T, T)

        # 5. Aggregate the Values
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, head_size)

        return out

# --- Sanity Check ---
# Let's create some fake data to test the dimensions
dummy_x = torch.randn(4, 8, 32) # Batch=4, Time=8, Channels=32
attention_head = Head(head_size=16)
output = attention_head(dummy_x)

print(f"Input shape: {dummy_x.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([4, 8, 32])
Output shape: torch.Size([4, 8, 16])


In [22]:
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        # Create a list of our individual Attention Heads
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

        # A linear layer to mix the outputs of all heads back together
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        # Run all heads in parallel and concatenate them along the channel dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [23]:
class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # In standard Transformers, the hidden layer is 4x the embedding size
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            # Projection layer to output back to the original dimension
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.net(x)

In [24]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        # How big is each head? Total embedding size divided by number of heads
        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        # Layer Normalizations
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Notice the residual connection: x = x + layer(norm(x))
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [15]:
# --- 1. The Final GPT Model Architecture ---
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size):
        super().__init__()
        # The token identities
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # The token positions (The "name tags")
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # A sequence of our Transformer Blocks stacked on top of each other
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])

        # Final Layer Normalization and Linear layer to output our vocabulary scores
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Extract embeddings for the tokens and their specific positions
        tok_emb = self.token_embedding_table(idx) # (Batch, Time, Channels)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (Time, Channels)

        # The core logic: Add identity and position together, then pass to the Transformer blocks
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # The inference loop for generating text
        for _ in range(max_new_tokens):
            # Crucial: Crop the context to the last block_size tokens so we don't overflow the positional embeddings
            idx_cond = idx[:, -block_size:]

            # Forward pass
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


# --- 2. Initialize the Model and Hyperparameters ---
batch_size = 32
block_size = 8
max_iters = 5000
learning_rate = 1e-3
n_embd = 32
n_head = 4
n_layer = 3  # We are stacking 3 Transformer blocks

# Build the model and send it to the GPU
model = GPTLanguageModel(vocab_size, n_embd, n_head, n_layer, block_size)
model = model.to(device)

print(f"Model built with {sum(p.numel() for p in model.parameters())/1e6:.3f} Million parameters.")


# --- 3. The Big Training Run ---
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("\nStarting the training run...")
for iter in range(max_iters):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 500 == 0:
        print(f"Step {iter:4d} | Loss: {loss.item():.4f}")

print(f"Final Loss: {loss.item():.4f}")


# --- 4. The Magic: Generating True Shakespeare ---
print("\n--- NanoGPT Output ---\n")
# Start with a single newline character
context = torch.zeros((1, 1), dtype=torch.long, device=device)
# Generate 500 characters
generated_tokens = model.generate(context, max_new_tokens=500)[0].tolist()
print(decode(generated_tokens))

Model built with 0.042 Million parameters.

Starting the training run...
Step    0 | Loss: 4.3724
Step  500 | Loss: 2.6197
Step 1000 | Loss: 2.1901
Step 1500 | Loss: 2.1359
Step 2000 | Loss: 2.2751
Step 2500 | Loss: 1.9921
Step 3000 | Loss: 2.0089
Step 3500 | Loss: 2.0999
Step 4000 | Loss: 2.1184
Step 4500 | Loss: 1.9708
Final Loss: 1.8957

--- NanoGPT Output ---



KING CLINGNEY:
Titjestiol:
Eard, I sto deay I bettritate graint pindy?
Sonaul's Nade thosesth:
Sain thear wheart his hath fa't fure my leif;
And befimese mager'd:
Sisons you how come bral holdss.

FLUSTET:
First it, Glall by shad, angual I wauliving and as my I hen resiff, pecterceming thou had? whill Eleal come semning of imamad:
; to how him holsort thee to lenanten noed
To the collis they be nesty shate
And they sair. Worlo to duke oskings:
Liers plot
Onlseign Edpancesenceo?

KECKING-BETBET:
